In [21]:
# Pipeline aktif: SVM baseline dan Transformer dibandingkan
# dengan Fuzzy state yang sama.

from pathlib import Path
import joblib
import sys
import numpy as np

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "modules").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from modules.hostage_fuzzy import (
    calculate_ai_defense_score,
    describe_ai_defense_intent,
)

from modules.hostage_nlg import (
    GameMemory,
    generate_npc_response,
    build_nlg_provider,
)

from modules.nlu_training import predict_transformer_intent


# ============================================================
# NLG PROVIDER
# ============================================================

llm = build_nlg_provider()


# ============================================================
# GLOBAL STATE
# ============================================================

GAME_MEMORY = GameMemory()
SVM_MODEL = None


# ============================================================
# MODEL DIRECTORY
# ============================================================

def _model_dir():
    for base in (Path.cwd(), PROJECT_ROOT):
        candidate = base / "models"

        if (candidate / "intent_classifier_svm.pkl").is_file():
            return candidate

    raise FileNotFoundError(
        "Model SVM baseline belum tersedia. "
        "Jalankan notebook NLU terlebih dahulu."
    )


# ============================================================
# SVM
# ============================================================

def get_svm_intent_with_confidence(user_text):
    """Prediksi intent SVM + confidence."""

    global SVM_MODEL

    if SVM_MODEL is None:
        SVM_MODEL = joblib.load(
            _model_dir() / "intent_classifier_svm.pkl"
        )

    predicted = str(
        SVM_MODEL.predict([user_text])[0]
    )

    if hasattr(SVM_MODEL, "predict_proba"):
        confidence = float(
            np.max(
                SVM_MODEL.predict_proba([user_text])[0]
            ) * 100
        )
    else:
        confidence = float("nan")

    return predicted, round(confidence, 2)


# ============================================================
# TRANSFORMER
# ============================================================

def get_transformer_intent(user_text):
    return predict_transformer_intent(
        user_text,
        _model_dir(),
        device="cpu"
    )


# ============================================================
# FUZZY
# ============================================================

def get_fuzzy_intent(persentase_dituduh):
    score = calculate_ai_defense_score(
        persentase_dituduh
    )

    intent = describe_ai_defense_intent(score)

    return intent, round(score, 2)


# ============================================================
# ENSEMBLE
# ============================================================

def resolve_intent_ensemble(
    user_text: str,
    persentase_dituduh: float = 0.0,
    confidence_floor: float = 55.0,
) -> dict:

    svm_intent, svm_conf = (
        get_svm_intent_with_confidence(user_text)
    )

    transformer_intent, transformer_conf = (
        get_transformer_intent(user_text)
    )

    # --------------------------------------------------------
    # Kalau salah satu model cukup yakin
    # --------------------------------------------------------

    if (
        svm_conf >= confidence_floor
        or transformer_conf >= confidence_floor
    ):

        if svm_conf >= transformer_conf:
            final_intent = svm_intent
            final_conf = svm_conf
            source = "svm"

        else:
            final_intent = transformer_intent
            final_conf = transformer_conf
            source = "transformer"

        fuzzy_intent, fuzzy_score = get_fuzzy_intent(
            persentase_dituduh
        )

    # --------------------------------------------------------
    # Kalau dua-duanya ragu → Fuzzy
    # --------------------------------------------------------

    else:

        fuzzy_intent, fuzzy_score = get_fuzzy_intent(
            persentase_dituduh
        )

        final_intent = fuzzy_intent
        final_conf = None
        source = "fuzzy_fallback"

    return {
        "final_intent": final_intent,
        "final_confidence": final_conf,
        "source": source,

        "svm_intent": svm_intent,
        "svm_confidence": svm_conf,

        "transformer_intent": transformer_intent,
        "transformer_confidence": transformer_conf,

        "fuzzy_intent": fuzzy_intent,
        "fuzzy_score": fuzzy_score,
    }


# ============================================================
# MEMORY
# ============================================================

def reset_game_memory():
    GAME_MEMORY.entries.clear()


# ============================================================
# NLG
# ============================================================

def npc_respond_with_intent(
    chat_pemain,
    intent,
    game_state,
    npc_name="NPC",
    speaker_pemain="Pemain",
):

    return generate_npc_response(
        llm=llm,
        chat_pemain=chat_pemain,
        intent=intent,
        game_state=game_state,
        npc_name=npc_name,
        memory=GAME_MEMORY,
        speaker_pemain=speaker_pemain,
    )

In [22]:
# =============================================================================
# Test chat untuk SEMUA intent — SVM baseline vs Transformer
# fase siang vs malam
# =============================================================================

TEST_CASES = [
    ("offend",  "B kena Gag Order ketika mulai ditanya alibinya, menurutku itu pola Hitman."),
    ("defend",  "Aku bukan Hitman. Tuduhan itu tidak punya bukti dari chat publik."),
    ("defend",  "Aku Spy dan semalam Guard Raka, jadi jangan curigai dia."),
    ("neutral", "Stalker, kamu Peek siapa semalam dan apa hasil yang kamu lihat?"),
    ("offend",  "Jangan hanya fokus ke aku; cek D yang ceritanya berubah setiap ditanya."),
    ("offend",  "Vote C saja, dia paling diuntungkan ketika seseorang mendadak diam."),
    ("defend",  "Klaimku Civilian, jadi aku memang tidak memiliki aksi malam."),
    ("neutral", "Malam ini chat terkunci, kita lanjut diskusi setelah fase pagi."),
    ("offend",  "A selalu menghindar saat kita tanya kenapa dia menuduh tanpa bukti."),
    ("defend",  "Diamku bukan pengakuan; aku masih bisa menjelaskan alibiku di fase siang."),
]


BASE_GAME_STATE = {
    "phase": "diskusi",
    "round": 3,
    "persentase_dituduh": 40,
    "public_events": [
        "Gag Order dipakai saat diskusi.",
        "Tidak ada target Hostage yang diumumkan sistem.",
    ],
    "silent_players": ["B"],
}


def run_single_case(
    expected_intent: str,
    chat: str,
    npc_name: str = "Naya",
    speaker: str = "Raka",
):
    """Jalankan 1 kalimat lewat SVM, Transformer, dan fase malam."""

    # =========================================================
    # RESET MEMORY
    # =========================================================

    reset_game_memory()

    GAME_MEMORY.add(
        speaker,
        chat,
        phase="diskusi",
        round_number=BASE_GAME_STATE["round"],
        target=npc_name,
    )


    # =========================================================
    # FUZZY
    # =========================================================

    fuzzy_score = calculate_ai_defense_score(
        BASE_GAME_STATE["persentase_dituduh"]
    )

    fuzzy_intent = describe_ai_defense_intent(
        fuzzy_score
    )


    # =========================================================
    # SVM BASELINE
    # =========================================================

    svm_intent, svm_confidence = (
        get_svm_intent_with_confidence(chat)
    )

    svm_result = npc_respond_with_intent(
        chat,
        svm_intent,
        BASE_GAME_STATE,
        npc_name=npc_name,
        speaker_pemain=speaker,
    )


    # =========================================================
    # TRANSFORMER
    # =========================================================

    reset_game_memory()

    transformer_intent, transformer_confidence = (
        get_transformer_intent(chat)
    )

    transformer_result = npc_respond_with_intent(
        chat,
        transformer_intent,
        BASE_GAME_STATE,
        npc_name=npc_name,
        speaker_pemain=speaker,
    )


    # =========================================================
    # FASE MALAM
    # =========================================================

    night_result = npc_respond_with_intent(
        chat,
        svm_intent,
        {
            **BASE_GAME_STATE,
            "phase": "malam",
        },
        npc_name=npc_name,
        speaker_pemain=speaker,
    )


    # =========================================================
    # RETURN
    # =========================================================

    return {
        "expected_intent": expected_intent,
        "chat": chat,

        # Fuzzy
        "fuzzy_score": fuzzy_score,
        "fuzzy_intent": fuzzy_intent,

        # SVM
        "svm_intent": svm_intent,
        "svm_confidence": svm_confidence,
        "svm_match": svm_intent == expected_intent,
        "svm_nlg": svm_result.get("nlg_provider"),
        "svm_reply": svm_result.get("npc_reply"),

        # Transformer
        "transformer_intent": transformer_intent,
        "transformer_confidence": transformer_confidence,
        "transformer_match": transformer_intent == expected_intent,
        "transformer_nlg": transformer_result.get("nlg_provider"),
        "transformer_reply": transformer_result.get("npc_reply"),

        # Night
        "night_locked": not night_result.get("response_allowed"),
        "night_system_message": night_result.get("system_message"),
    }


def run_all_test_cases(
    cases=TEST_CASES,
    verbose: bool = True,
):
    """Jalankan semua test case."""

    results = []

    for idx, (expected_intent, chat) in enumerate(
        cases,
        start=1,
    ):

        r = run_single_case(
            expected_intent,
            chat,
        )

        results.append(r)

        if verbose:

            svm_flag = "OK" if r["svm_match"] else "MISS"
            trf_flag = "OK" if r["transformer_match"] else "MISS"

            print(f"--- Kasus {idx} [{expected_intent}] ---")

            print(
                f"Chat: {r['chat']}"
            )

            print(
                f"Fuzzy       -> "
                f"{r['fuzzy_intent']:<8} "
                f"(score={r['fuzzy_score']:.2f})"
            )

            print(
                f"SVM         [{svm_flag}] -> "
                f"{r['svm_intent']:<8} "
                f"({r['svm_confidence']:.2f}%) | "
                f"NLG={r['svm_nlg']} | "
                f"NPC: {r['svm_reply']}"
            )

            print(
                f"Transformer  [{trf_flag}] -> "
                f"{r['transformer_intent']:<8} "
                f"({r['transformer_confidence']:.2f}%) | "
                f"NLG={r['transformer_nlg']} | "
                f"NPC: {r['transformer_reply']}"
            )

            print(
                f"Fase malam terkunci: "
                f"{r['night_locked']} | "
                f"{r['night_system_message']}"
            )

            print()

    return results


def summarize_accuracy(results):
    """Ringkasan akurasi SVM dan Transformer."""

    total = len(results)

    svm_correct = sum(
        r["svm_match"]
        for r in results
    )

    trf_correct = sum(
        r["transformer_match"]
        for r in results
    )

    print("=" * 65)

    print(
        f"Akurasi SVM         : "
        f"{svm_correct}/{total} "
        f"({svm_correct / total * 100:.1f}%)"
    )

    print(
        f"Akurasi Transformer : "
        f"{trf_correct}/{total} "
        f"({trf_correct / total * 100:.1f}%)"
    )

    print("=" * 65)

    intents = sorted(
        set(
            r["expected_intent"]
            for r in results
        )
    )

    print(
        f"{'Intent':<14}"
        f"{'SVM benar':<15}"
        f"{'Transformer benar':<20}"
        f"{'n':<5}"
    )

    for intent in intents:

        subset = [
            r for r in results
            if r["expected_intent"] == intent
        ]

        n = len(subset)

        svm_n = sum(
            r["svm_match"]
            for r in subset
        )

        trf_n = sum(
            r["transformer_match"]
            for r in subset
        )

        print(
            f"{intent:<14}"
            f"{svm_n}/{n:<13}"
            f"{trf_n}/{n:<18}"
            f"{n:<5}"
        )

    # =========================================================
    # MISMATCH
    # =========================================================

    mismatches = [
        r
        for r in results
        if (
            not r["svm_match"]
            or not r["transformer_match"]
        )
    ]

    if mismatches:

        print("\nKasus mismatch:")

        for r in mismatches:

            print(
                f"  expected={r['expected_intent']:<8} "
                f"SVM={r['svm_intent']:<8} "
                f"Transformer={r['transformer_intent']:<8}"
            )

            print(
                f"  \"{r['chat']}\""
            )

    else:

        print(
            "\nSemua kasus cocok dengan expected_intent."
        )


# =============================================================
# RUN
# =============================================================

all_results = run_all_test_cases()

summarize_accuracy(all_results)

--- Kasus 1 [offend] ---
Chat: B kena Gag Order ketika mulai ditanya alibinya, menurutku itu pola Hitman.
Fuzzy       -> offend   (score=50.00)
SVM         [OK] -> offend   (97.73%) | NLG=openrouter_qwen14b | NPC: Gw butuh lihat alur chat dan bukti publik dulu sebelum ambil sikap.
Transformer  [OK] -> offend   (99.91%) | NLG=openrouter_qwen14b | NPC: Gw butuh lihat alur chat dan bukti publik dulu sebelum ambil sikap.
Fase malam terkunci: True | Chat terkunci selama fase malam.

--- Kasus 2 [defend] ---
Chat: Aku bukan Hitman. Tuduhan itu tidak punya bukti dari chat publik.
Fuzzy       -> offend   (score=50.00)
SVM         [OK] -> defend   (95.70%) | NLG=openrouter_qwen14b | NPC: Gw butuh lihat alur chat dan bukti publik dulu sebelum ambil sikap.
Transformer  [OK] -> defend   (99.46%) | NLG=openrouter_qwen14b | NPC: Gw butuh lihat alur chat dan bukti publik dulu sebelum ambil sikap.
Fase malam terkunci: True | Chat terkunci selama fase malam.

--- Kasus 3 [defend] ---
Chat: Aku Spy dan 